# Первое задание по курсовой работе

In [ ]:
!pip install darts prophet catboost plotly

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from darts import TimeSeries
from darts.models import Prophet as DartsProphet, CatBoostModel
from darts.metrics import mae, rmse
import warnings
warnings.filterwarnings('ignore')

In [4]:
# загрузка данных
df = pd.read_csv('timeseries_hse_teducation_df.csv')
df['dttm_30'] = pd.to_datetime(df['dttm_30'])
df = df.sort_values(['ts_name', 'dttm_30']).reset_index(drop=True)

In [5]:
print(f"Уникальных рядов: {df['ts_name'].nunique()}")
print(f"Диапазон дат: {df['dttm_30'].min()} - {df['dttm_30'].max()}")

Уникальных рядов: 12
Диапазон дат: 2023-01-01 00:00:00 - 2025-12-20 23:30:00


In [8]:
# функция для работы с отдельным временным рядом
def create_timeseries(data, ts_name):
    ts_data = data[data['ts_name'] == ts_name][['dttm_30', 'y']].copy()
    ts_data = ts_data.drop_duplicates('dttm_30').set_index('dttm_30')

    # заполняем пропуски, так как нам важна регулярность в данных
    all_dates = pd.date_range(start=ts_data.index.min(), end=ts_data.index.max(), freq='30T')
    ts_data = ts_data.reindex(all_dates).ffill() # метод forward fill (ffill)

    series = TimeSeries.from_dataframe(
        ts_data.reset_index(),
        time_col='index',
        value_cols='y',
        freq='30T'
    )
    return series

In [9]:
# train/test

test_start_date = pd.Timestamp('2025-09-01')
forecast_days = 60
forecast_steps = forecast_days * 24 * 2 # 2880 шагов

# выберу один временной ряд
ts_name = 'ts_name_0'
series = create_timeseries(df, ts_name)

# разделение
train_series, test_series_full = series.split_before(test_start_date)
test_series = test_series_full[:forecast_steps]

## Обучение моделей и прогноз

In [10]:
# Prophet
print("Обучение Prophet...")
prophet_model = DartsProphet()
prophet_model.fit(train_series)
prophet_forecast = prophet_model.predict(n=len(test_series))

# CatBoost
print("Обучение CatBoost...")
catboost_model = CatBoostModel(
    lags=48, # лаг - 24 часа (30мин * 48)
    output_chunk_length=1,
    random_state=42,
    verbose=False
)
catboost_model.fit(train_series)
catboost_forecast = catboost_model.predict(n=len(test_series))

Обучение Prophet...
Обучение CatBoost...


## Расчет метрик

In [11]:
metrics_df = pd.DataFrame({
    'Model': ['Prophet', 'CatBoost'],
    'MAE': [mae(test_series, prophet_forecast), mae(test_series, catboost_forecast)],
    'RMSE': [rmse(test_series, prophet_forecast), rmse(test_series, catboost_forecast)]
})

display(metrics_df)

,Model,MAE,RMSE
0,Prophet,86.192154,108.539241
1,CatBoost,65.550243,98.211685


## Визуализация

In [12]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_forecast(train, test, p_fc, c_fc, name):
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(f"Прогноз Prophet для {name}", f"Прогноз CatBoost для {name}")
    )

    # последние 30 дней трейна для контекста
    train_recent = train.tail(24 * 2 * 30)

    # график Prophet
    # train
    fig.add_trace(go.Scatter(
        x=train_recent.time_index, y=train_recent.values().flatten(),
        name='Train (Last 30d)', line=dict(color='blue', width=1),
        legendgroup="train", showlegend=True
    ), row=1, col=1)

    # test
    fig.add_trace(go.Scatter(
        x=test.time_index, y=test.values().flatten(),
        name='Actual (Test)', line=dict(color='black', width=2),
        legendgroup="fact", showlegend=True
    ), row=1, col=1)

    # predict Prophet
    fig.add_trace(go.Scatter(
        x=p_fc.time_index, y=p_fc.values().flatten(),
        name='Prophet Forecast', line=dict(color='red', dash='dash'),
        legendgroup="prophet", showlegend=True
    ), row=1, col=1)

    # график catboost
    # train
    fig.add_trace(go.Scatter(
        x=train_recent.time_index, y=train_recent.values().flatten(),
        name='Train (Last 30d)', line=dict(color='blue', width=1),
        legendgroup="train", showlegend=False
    ), row=2, col=1)

    # test
    fig.add_trace(go.Scatter(
        x=test.time_index, y=test.values().flatten(),
        name='Actual (Test)', line=dict(color='black', width=2),
        legendgroup="fact", showlegend=False
    ), row=2, col=1)

    # predict CatBoost
    fig.add_trace(go.Scatter(
        x=c_fc.time_index, y=c_fc.values().flatten(),
        name='CatBoost Forecast', line=dict(color='green', dash='dot'),
        legendgroup="catboost", showlegend=True
    ), row=2, col=1)

    fig.update_layout(
        height=800,
        template='plotly_white',
        hovermode='x unified',
        title_text=f"Сравнение моделей прогнозирования для {name}",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.update_yaxes(title_text="Значение", row=1, col=1)
    fig.update_yaxes(title_text="Значение", row=2, col=1)
    fig.update_xaxes(title_text="Дата", row=2, col=1)

    fig.show()
plot_forecast(train_series, test_series, prophet_forecast, catboost_forecast, ts_name)
